In [1]:
%load_ext cudf.pandas
%load_ext cuml.accel

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

DEVICE = 'cuda' if torch.cuda.is_available else 'cpu'
DEVICE

/home/dex/miniconda3/envs/rapids_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cuda'

In [2]:
df = pd.read_csv('datasets/Quora Questions Pair Dataset/10_Xy.csv')
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [4]:
embed_matrix = torch.from_numpy(np.loadtxt('datasets/Quora Questions Pair Dataset/07_emb_matrix.csv',
                                       delimiter=","))
embed_matrix

tensor([[ 0.3756,  0.2267, -0.0065,  ..., -0.2938,  0.4096,  1.8223],
        [ 0.0036,  0.0667,  0.4862,  ..., -0.1020,  0.0392, -0.2243],
        [ 0.2502,  0.0046,  1.3010,  ...,  0.1616, -0.0993, -0.0726],
        ...,
        [ 0.9871,  0.8178, -0.6882,  ..., -1.6554, -2.7404, -0.2003],
        [ 0.7834,  2.6968,  2.5663,  ..., -1.0214,  2.3385,  0.3786],
        [-1.4707,  0.7946,  0.0672,  ..., -0.9628,  1.7868, -1.3583]],
       dtype=torch.float64)

In [5]:
q1_ids = np.loadtxt('datasets/Quora Questions Pair Dataset/04_q1_ids.csv', delimiter=",")
q2_ids = np.loadtxt('datasets/Quora Questions Pair Dataset/05_q2_ids.csv', delimiter=",")

In [6]:
class CustomDataset(Dataset):
    def __init__(self, q1, q2, y):
        self.q1 = torch.tensor(q1, dtype=torch.long)
        self.q2 = torch.tensor(q2, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.q1[idx], self.q2[idx], self.y[idx]

In [7]:
class model_gru(nn.Module):
    def __init__(self, embed_weight, hidden_dim=128):
        super().__init__()
        vocab_size, embed_dim = embed_weight.shape
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.embed.weight.data.copy_(embed_weight)
        self.embed.weight.requires_grad = True

        self.gru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(
            nn.Linear(hidden_dim*4+2, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128,1)
        )
    def encode(self, X):
        embed = self.embed(X)
        _, h = self.gru(embed)
        h = torch.cat([h[0], h[1]], dim=1)
        return h
    def forward(self, q1, q2):
        h1, h2 = self.encode(q1), self.encode(q2)
        cos = F.cosine_similarity(h1, h2).unsqueeze(1)
        l1 = torch.abs(h1 - h2).sum(1, keepdim=True)
        combined = torch.cat([h1, h2, cos, l1], dim=1)
        return self.fc(combined).squeeze(1)

In [8]:
train_ds = CustomDataset(q1_ids[:len(X_train)], q2_ids[:len(X_train)], y[:len(X_train)])

In [9]:
train_loader = DataLoader(train_ds, batch_size=512, shuffle=True)

In [10]:
#DEVICE = 'cpu'

In [11]:
model = model_gru(embed_matrix.cpu()).to(DEVICE)

In [12]:
opt = torch.optim.Adam(model.parameters(), lr=1e-3)

In [13]:
criterion = nn.BCEWithLogitsLoss()

In [ ]:
for epoch in range(50):
    total = 0
    for q1b, q2b, yb in tqdm(train_loader):
        q1b, q2b, yb = q1b.to(DEVICE), q2b.to(DEVICE), yb.to(DEVICE)
        logits = model(q1b, q2b)
        loss = criterion(logits, yb)
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    print(f'Epoch {epoch+1} loss={total/len(train_loader):.4f}')

100%|████████████████████████████████████████████████████████████████████████████████| 632/632 [00:10<00:00, 58.29it/s]


Epoch 1 loss=0.5441


100%|████████████████████████████████████████████████████████████████████████████████| 632/632 [00:10<00:00, 62.99it/s]


Epoch 2 loss=0.5010


100%|████████████████████████████████████████████████████████████████████████████████| 632/632 [00:09<00:00, 63.82it/s]


...

100%|████████████████████████████████████████████████████████████████████████████████| 632/632 [00:10<00:00, 59.15it/s]


Epoch 30 loss=0.0780


100%|████████████████████████████████████████████████████████████████████████████████| 632/632 [00:10<00:00, 59.23it/s]


Epoch 31 loss=0.0748


 12%|█████████▍                                                                       | 74/632 [00:01<00:09, 59.93it/s]

In [ ]:
torch.save(model, "models/gru_model.pth")

In [ ]:
model = torch.load("models/gru_model.pth")

## test / Evaluation

In [ ]:
model.eval()

In [ ]:
test_ds = CustomDataset(q1_ids[len(X_train):], q2_ids[len(X_train):], y[len(X_train):])

In [ ]:
test_loader = DataLoader(test_ds, batch_size=512, shuffle=False)

In [ ]:
all_probs = []
all_labels = []
with torch.no_grad():
    for q1b, q2b, yb in test_loader:
        q1b, q2b = q1b.to(DEVICE), q2b.to(DEVICE)
        logits = model(q1b, q2b)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs)
        all_labels.append(yb.cpu().numpy())

In [ ]:
q1_ids[len(X_train):].shape, q2_ids[len(X_train):].shape, y[len(X_train):].shape

In [ ]:
80794/512, len(test_loader), 158 * 512

In [ ]:
len(all_probs) * len(all_probs[0])

In [ ]:
len(all_labels) * len(all_labels[0])

In [ ]:
all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)

In [ ]:
all_probs.shape, all_probs.shape

In [ ]:
all_preds = (all_probs >= 0.5).astype(int)

In [ ]:
accuracy_score(all_labels, all_preds), precision_score(all_labels, all_preds), recall_score(all_labels, all_preds), f1_score(all_labels, all_preds), confusion_matrix(all_labels, all_preds)

In [ ]:
all_preds = (all_probs >= 0.4).astype(int)

In [ ]:
accuracy_score(all_labels, all_preds), precision_score(all_labels, all_preds), recall_score(all_labels, all_preds), f1_score(all_labels, all_preds), confusion_matrix(all_labels, all_preds)

In [ ]:
all_preds = (all_probs >= 0.6).astype(int)

In [ ]:
accuracy_score(all_labels, all_preds), precision_score(all_labels, all_preds), recall_score(all_labels, all_preds), f1_score(all_labels, all_preds), confusion_matrix(all_labels, all_preds)